# Notebook 02 — Reddit Scraping
**Owner:** Person 2

**Goal:** Pull Reddit posts and comments from hardware communities filtered by keywords and date range.

**Output:** `../data/reddit/reddit_raw.csv`

**Columns:** `post_id | date | subreddit | title | body | top_comments | upvotes | url`

---
**Setup:** Register a Reddit app at https://www.reddit.com/prefs/apps (choose 'script' type).  
Fill in your credentials in the config cell below. Do NOT commit credentials to GitHub — use a `.env` file or enter them manually each session.

In [ ]:
import praw
import pandas as pd
from datetime import datetime, timezone
import time

## 1. Reddit API Credentials
Get these from https://www.reddit.com/prefs/apps — create a 'script' type app.

In [ ]:
# Fill these in — do NOT push credentials to GitHub
REDDIT_CLIENT_ID = "YOUR_CLIENT_ID"
REDDIT_CLIENT_SECRET = "YOUR_CLIENT_SECRET"
REDDIT_USER_AGENT = "hardware_price_tracker/0.1 by YOUR_USERNAME"

reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT
)
print("Connected (read-only):", reddit.read_only)

## 2. Define Search Parameters

In [ ]:
SUBREDDITS = ["buildapc", "hardware", "pcmasterrace"]

KEYWORDS = [
    "RAM price", "DDR5 price", "memory price",
    "GPU price", "graphics card price", "RTX 4060 price",
    "hardware prices", "PC parts price"
]

# Align with the date range of your price data
DATE_START = datetime(2024, 1, 1, tzinfo=timezone.utc)
DATE_END   = datetime(2026, 4, 9, tzinfo=timezone.utc)

MAX_POSTS_PER_QUERY = 100  # PRAW max per request; use pushshift for deeper history

print(f"Date range: {DATE_START.date()} → {DATE_END.date()}")

## 3. Scrape Posts

In [ ]:
def get_top_comments(submission, n=3) -> str:
    """Return top N comments joined as a single string."""
    submission.comments.replace_more(limit=0)
    comments = []
    for comment in list(submission.comments)[:n]:
        if hasattr(comment, 'body'):
            comments.append(comment.body)
    return " | ".join(comments)


def scrape_subreddit_keyword(subreddit_name: str, keyword: str, limit: int = 100) -> list:
    records = []
    subreddit = reddit.subreddit(subreddit_name)
    
    for submission in subreddit.search(keyword, sort="new", limit=limit):
        post_date = datetime.fromtimestamp(submission.created_utc, tz=timezone.utc)
        
        if not (DATE_START <= post_date <= DATE_END):
            continue
        
        records.append({
            "post_id":      submission.id,
            "date":         post_date.strftime("%Y-%m-%d"),
            "subreddit":    subreddit_name,
            "title":        submission.title,
            "body":         submission.selftext,
            "top_comments": get_top_comments(submission),
            "upvotes":      submission.score,
            "url":          f"https://reddit.com{submission.permalink}"
        })
    
    return records


print("Functions defined — ready to scrape")

In [ ]:
all_records = []

for subreddit in SUBREDDITS:
    for keyword in KEYWORDS:
        print(f"  r/{subreddit} | '{keyword}'...", end=" ")
        records = scrape_subreddit_keyword(subreddit, keyword, limit=MAX_POSTS_PER_QUERY)
        all_records.extend(records)
        print(f"{len(records)} posts")
        time.sleep(0.5)  # rate limit

df_raw = pd.DataFrame(all_records).drop_duplicates(subset=["post_id"])
print(f"\nTotal unique posts collected: {len(df_raw)}")
df_raw.head()

## 4. Save Raw Data

In [ ]:
OUTPUT_PATH = "../data/reddit/reddit_raw.csv"
df_raw.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
print(df_raw.dtypes)
print(df_raw["subreddit"].value_counts())

## 5. Quick Sanity Check

In [ ]:
import matplotlib.pyplot as plt

df_raw["date"] = pd.to_datetime(df_raw["date"])
df_raw.set_index("date")["post_id"].resample("W").count().plot(
    title="Posts per Week", figsize=(12, 4), ylabel="Post Count"
)
plt.tight_layout()
plt.show()